In [ ]:
from IPython.display import display, HTML, Markdown

title_html = """
<div style=\"text-align: center; margin: 20px 0;\">
    <h1 style=\"color: #2E7D32; margin-bottom: 10px;\">
        💊 Medication Reminder Chatbot
    </h1>
    <h3 style=\"color: #558B2F; margin-bottom: 20px;\">
        FDA-Grounded Medical Information System
    </h3>
    <p style=\"font-size: 14px; color: #666;\">
        GenAIVersity 24-Hour Hackathon | Team: Data & LLM Engineers
    </p>
</div>
"""
display(HTML(title_html))

In [ ]:
import sys
sys.path.insert(0, '../')

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, JSON
import json
from datetime import datetime, timedelta
import time

# Import our modules (these will be created by Members 1, 2, 3)
try:
    from src.retrieval import get_retriever
    from src.generation import get_generator
    from src.guardrails import get_guardrails
    from src.utils import log_event, print_header
    print("✅ All modules imported successfully")
except ImportError as e:
    print(f"⚠ Import error: {str(e)}")
    print("Make sure Members 1, 2, 3 have committed their code")

In [ ]:
# Custom CSS for nice formatting
STYLE_CSS = """
<style>
.success-box { 
    background-color: #d4edda; 
    border: 2px solid #28a745; 
    padding: 15px; 
    border-radius: 5px;
    margin: 10px 0;
    font-weight: 500;
}
.warning-box { 
    background-color: #fff3cd; 
    border: 2px solid #ffc107; 
    padding: 15px; 
    border-radius: 5px;
    margin: 10px 0;
    font-weight: 500;
}
.danger-box { 
    background-color: #f8d7da; 
    border: 2px solid #dc3545; 
    padding: 15px; 
    border-radius: 5px;
    margin: 10px 0;
    font-weight: 500;
}
.info-box {
    background-color: #d1ecf1;
    border: 2px solid #0c5460;
    padding: 15px;
    border-radius: 5px;
    margin: 10px 0;
}
.section-header {
    background-color: #2E7D32;
    color: white;
    padding: 15px;
    border-radius: 5px;
    margin-top: 20px;
    margin-bottom: 15px;
    font-size: 18px;
    font-weight: bold;
}
.metric-box {
    background-color: #f5f5f5;
    border-left: 4px solid #2E7D32;
    padding: 10px;
    margin: 5px 0;
    font-family: monospace;
}
.output-display {
    background-color: #f9f9f9;
    border: 1px solid #ddd;
    padding: 10px;
    border-radius: 3px;
    margin: 5px 0;
    max-height: 300px;
    overflow-y: auto;
    font-family: monospace;
    font-size: 12px;
}
</style>
"""
display(HTML(STYLE_CSS))

In [ ]:
print("Initializing systems...\n")

# Initialize retriever
try:
    retriever = get_retriever()
    stats = retriever.get_collection_stats()
    print(f"✅ Retrieval System Ready")
    print(f"    └─ Documents indexed: {stats['total_documents']}")
    print(f"    └─ Collection: {stats['collection_name']}")
except Exception as e:
    print(f"⚠ Retrieval system error: {str(e)}")
    retriever = None

# Initialize generator
try:
    generator = get_generator()
    print(f"✅ LLM Generation Ready")
except Exception as e:
    print(f"⚠ Generation system error: {str(e)}")
    generator = None

# Initialize guardrails
try:
    guardrails = get_guardrails(retriever)
    print(f"✅ Safety Guardrails Ready")
except Exception as e:
    print(f"⚠ Guardrails error: {str(e)}")
    guardrails = None

# Get drug list for dropdowns
try:
    drug_names = retriever.get_drug_names()
    drug_names_list = sorted(drug_names)[:100]  # Top 100 for UI
    print(f"✅ Loaded {len(drug_names_list)} drugs")
except Exception as e:
    print(f"⚠ Error loading drugs: {str(e)}")
    drug_names_list = []

print("\n✅ All systems initialized and ready!\n")

In [ ]:
display(HTML('<div class="section-header">🔍 SECTION 1: Drug Information Lookup</div>'))

# Create UI components
query_input = widgets.Text(
    placeholder="Ask about a medication (e.g., 'What is metformin used for?')",
    description="Your Question:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='80%')
)

ask_button = widgets.Button(
    description="Ask",
    button_style='success',
    tooltip="Click to ask about a medication",
    layout=widgets.Layout(width='100px')
)

qa_output = widgets.Output()

def on_ask_click(b):
    """Handle ask button click"""
    with qa_output:
        qa_output.clear_output()
        query = query_input.value.strip()
        
        if not query:
            print("⚠ Please enter a question")
            return
        
        try:
            print(f"🔄 Processing: '{query}'\n")
            
            # Step 1: Retrieve
            print("1️⃣ Retrieving relevant information...")
            results = retriever.retrieve_chunks(query, n_results=10)
            retrieved_count = results['total_retrieved']
            print(f"    └─ Found {retrieved_count} relevant sections\n")
            
            # Step 2: Generate
            print("2️⃣ Generating response...")
            response = generator.generate_response(query, results['chunks'])
            print(f"    └─ Generated with confidence: {response['confidence']}\n")
            
            # Step 3: Validate Safety
            print("3️⃣ Running safety checks...")
            safety = guardrails.run_all_safety_checks(response, query, results['chunks'])
            print(f"    └─ Status: {safety['overall_safety_status']}\n")
            
            # Display results
            print("=" * 60)
            print("RESPONSE:")
            print("=" * 60)
            
            if safety['overall_safety_status'] == 'CRITICAL':
                display(HTML(f'<div class="danger-box">{safety["final_recommendation"]}</div>'))
            elif safety['overall_safety_status'] == 'WARNING':
                display(HTML(f'<div class="warning-box">{response["response"]}</div>'))
            else:
                display(HTML(f'<div class="success-box">{response["response"]}</div>'))
            
            # Display metadata
            print("\n" + "=" * 60)
            print("METADATA:")
            print("=" * 60)
            
            metadata_html = f"""
            <div class=\"metric-box\">
            <strong>Confidence Score:</strong> {response['confidence']:.2f} 
            {'✅' if response['confidence'] > 0.75 else '⚠' if response['confidence'] > 0.60 else '❌'}<br>
            <strong>Hallucination Risk:</strong> {safety['hallucination_check']['severity']}<br>
            <strong>Safety Status:</strong> {safety['overall_safety_status']}<br>
            <strong>Recommendation:</strong> {safety['final_recommendation']}
            </div>
            """
            display(HTML(metadata_html))
            
            # Display citations
            if response['citations']:
                print("\n" + "=" * 60)
                print("CITATIONS:")
                print("=" * 60)
                for i, citation in enumerate(response['citations'], 1):
                    print(f"{i}. {citation['section']} ({citation['drug'].title()})")
                    
        except Exception as e:
            display(HTML(f'<div class="danger-box">❌ Error: {str(e)}</div>'))

ask_button.on_click(on_ask_click)

# Display Section 1
display(Markdown("**Ask any question about medications. The system will answer based on FDA labels.**"))
display(query_input)
display(ask_button)
display(qa_output)

In [ ]:
display(HTML('<div class="section-header">⚠ SECTION 2: Drug Interaction Checker</div>'))

# Create drug dropdowns
drug1_dropdown = widgets.Dropdown(
    options=drug_names_list,
    value=drug_names_list[0] if drug_names_list else None,
    description="Drug 1:",
    style={'description_width': '100px'}
)

drug2_dropdown = widgets.Dropdown(
    options=drug_names_list,
    value=drug_names_list[1] if len(drug_names_list) > 1 else drug_names_list[0],
    description="Drug 2:",
    style={'description_width': '100px'}
)

check_button = widgets.Button(
    description="Check Interaction",
    button_style='warning',
    tooltip="Click to check for drug interactions"
)

interaction_output = widgets.Output()

def on_check_click(b):
    """Handle interaction check"""
    with interaction_output:
        interaction_output.clear_output()
        
        drug1 = drug1_dropdown.value
        drug2 = drug2_dropdown.value
        
        if drug1 == drug2:
            print("⚠ Please select two different drugs")
            return
        
        try:
            print(f"🔄 Checking: {drug1.title()} + {drug2.title()}\n")
            
            result = guardrails.check_interaction(drug1, drug2)
            
            print("=" * 60)
            print("RESULT:")
            print("=" * 60)
            
            if result['interaction_detected']:
                if result['severity'] == 'HIGH':
                    display(HTML(f'<div class="danger-box">🚫 {result["warning"]}</div>'))
                elif result['severity'] == 'MEDIUM':
                    display(HTML(f'<div class="warning-box">⚠ {result["warning"]}</div>'))
                else:
                    display(HTML(f'<div class="info-box">ℹ {result["warning"]}</div>'))
            else:
                display(HTML(f'<div class="success-box">✅ No dangerous interactions detected</div>'))
            
            print(f"\nSeverity: {result['severity']}")
            print(f"Recommendation: {result['recommendation']}")
            
            if result['evidence']:
                print("\nEvidence from FDA labels:")
                print(result['evidence'][:200] + "...")
                
        except Exception as e:
            display(HTML(f'<div class="danger-box">❌ Error: {str(e)}</div>'))

check_button.on_click(on_check_click)

# Display Section 2
display(Markdown("**Check if two drugs are safe to take together.**"))
display(drug1_dropdown)
display(drug2_dropdown)
display(check_button)
display(interaction_output)

In [ ]:
display(HTML('<div class="section-header">📅 SECTION 3: Reminder Generator</div>'))

# Create form fields
drug_reminder_input = widgets.Dropdown(
    options=drug_names_list,
    description="Drug:",
    style={'description_width': '100px'}
)

dosage_input = widgets.Text(
    placeholder="e.g., 500mg",
    description="Dosage:",
    style={'description_width': '100px'}
)

frequency_dropdown = widgets.Dropdown(
    options=["once daily", "twice daily", "three times daily", "every 4 hours", "every 6 hours", "every 8 hours"],
    description="Frequency:",
    style={'description_width': '100px'}
)

start_date_picker = widgets.DatePicker(
    description="Start Date:",
    value=datetime.now().date(),
    style={'description_width': '100px'}
)

duration_slider = widgets.IntSlider(
    min=1,
    max=90,
    value=30,
    description="Duration (days):",
    style={'description_width': '130px'}
)

generate_button = widgets.Button(
    description="Generate Reminder",
    button_style='info',
    tooltip="Click to generate medication reminder"
)

reminder_output = widgets.Output()

def on_generate_click(b):
    """Generate reminder schedule"""
    with reminder_output:
        reminder_output.clear_output()
        
        drug = drug_reminder_input.value
        dosage = dosage_input.value
        frequency = frequency_dropdown.value
        start_date = start_date_picker.value.strftime("%Y-%m-%d")
        duration = duration_slider.value
        
        if not dosage:
            print("⚠ Please enter a dosage")
            return
        
        try:
            print(f"🔄 Generating reminder...\n")
            
            reminder = generator.generate_reminder(
                drug=drug,
                dosage=dosage,
                frequency=frequency,
                start_date=start_date,
                duration_days=duration
            )
            
            print("=" * 60)
            print("REMINDER SCHEDULE GENERATED:")
            print("=" * 60)
            
            display(HTML(f"""
            <div class=\"success-box\">
            <strong>💊 Drug:</strong> {reminder['drug'].title()}<br>
            <strong>📊 Dosage:</strong> {reminder['dosage']}<br>
            <strong>⏰ Frequency:</strong> {reminder['frequency']}<br>
            <strong>📅 Period:</strong> {reminder['start_date']} to {reminder['end_date']}<br>
            <strong>📌 Total Doses:</strong> {reminder['total_doses']}
            </div>
            """))
            
            # Display schedule preview
            print("\n📋 Schedule Preview (first 10):")
            for i, time_slot in enumerate(reminder['schedule'][:10], 1):
                print(f"    {i}. {time_slot}")
            
            if len(reminder['schedule']) > 10:
                print(f"    ... and {len(reminder['schedule']) - 10} more")
            
            # Display warnings
            if reminder['warnings']:
                print("\n⚠ Important Warnings:")
                for warning in reminder['warnings']:
                    print(f"    • {warning}")
            
            # Show full JSON
            print("\n📥 Full Schedule (JSON):")
            print(json.dumps(reminder, indent=2))
            
        except Exception as e:
            display(HTML(f'<div class="danger-box">❌ Error: {str(e)}</div>'))

generate_button.on_click(on_generate_click)

# Display Section 3
display(Markdown("**Create a medication reminder schedule.**"))
display(drug_reminder_input)
display(dosage_input)
display(frequency_dropdown)
display(start_date_picker)
display(duration_slider)
display(generate_button)
display(reminder_output)

In [ ]:
display(HTML("""
<div style=\"margin-top: 30px; padding: 20px; background-color: #f0f7ff; border-left: 4px solid #0066cc;\">
    <h3>ℹ How to Use This Demo</h3>
    <ul>
        <li><strong>Section 1:</strong> Ask any question about medications. The system retrieves FDA labels and generates grounded responses.</li>
        <li><strong>Section 2:</strong> Check if two drugs are safe to take together. High-risk combinations are flagged.</li>
        <li><strong>Section 3:</strong> Generate a medication schedule with reminder times and important warnings.</li>
    </ul>
    <p><strong>Key Features:</strong></p>
    <ul>
        <li>✅ All information grounded in FDA drug labels</li>
        <li>✅ Safety checks detect hallucinations and dangerous interactions</li>
        <li>✅ Confidence scores indicate response reliability</li>
        <li>✅ Citations show data sources</li>
    </ul>
</div>
"""
))

print("\n✅ Demo Interface Ready!")